# Tennis Predictor — Elo Visualizations

Visualize:
1. Elo ratings over time for top players (Djokovic, Nadal, Federer, Sinner, Alcaraz)
2. Calibration curve (predicted vs actual win rates)
3. Accuracy over time (how does the model perform year by year?)
4. Distribution of model confidence on test set

**Prerequisites:**
- Backtest has been run and saved (`backtest_runs` has at least 1 entry)
- Elo snapshot exists for end date
- `pip install matplotlib seaborn` if not already

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from sqlalchemy import text

from tennis_predictor.data.storage import get_session
from tennis_predictor.backtest.walk_forward import run_walk_forward_backtest
from tennis_predictor.models.elo import EloConfig

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 100

## 1. Track Elo over time for top players

We need to rebuild Elo but save snapshots periodically. The simplest version: rerun build_elo at multiple end dates.

For exploration, here's a quicker hack: query the elo_ratings table for any snapshots we have saved.

In [ ]:
# Check what Elo snapshots we have available
with get_session() as session:
    snapshots = pd.read_sql(
        text("""
            SELECT rating_date, algorithm_version, COUNT(*) AS n_players
            FROM elo_ratings
            GROUP BY rating_date, algorithm_version
            ORDER BY rating_date
        """),
        session.bind,
    )
snapshots

In [ ]:
# Find the GOAT player IDs by name (fuzzy)
with get_session() as session:
    famous_players = pd.read_sql(
        text("""
            SELECT player_id, name_full, country_code
            FROM players
            WHERE tour = 'ATP'
              AND name_full IN ('Novak Djokovic', 'Rafael Nadal', 'Roger Federer',
                                'Andy Murray', 'Jannik Sinner', 'Carlos Alcaraz',
                                'Daniil Medvedev', 'Stefanos Tsitsipas')
        """),
        session.bind,
    )
famous_players

## 2. Re-run backtest in-memory (so we have predictions to plot)

Takes 5-8 min. Skip if you've already cached predictions.

In [ ]:
# This is the same backtest as before but we keep the predictions in memory
config = EloConfig()
summary, predictions = run_walk_forward_backtest(
    tour='ATP',
    train_start_date=date(2000, 1, 1),
    test_start_date=date(2011, 1, 1),
    test_end_date=date(2024, 12, 31),
    config=config,
    show_progress=True,
)
print(f'\nGenerated {len(predictions)} predictions')

In [ ]:
# Convert to dataframe for plotting
df = pd.DataFrame([
    {
        'date': p.match_date,
        'year': p.match_date.year,
        'surface': p.surface,
        'p_winner': p.p_winner_wins,
        'p_a': p.p_player_a_wins,
        'a_is_winner': p.a_is_winner,
    }
    for p in predictions
])
df['correct'] = ((df['p_a'] > 0.5) == df['a_is_winner']).astype(int)
df.head()

## 3. Calibration curve

X axis: predicted probability bucket
Y axis: actual fraction of wins in that bucket

A perfectly calibrated model lies on the diagonal.

In [ ]:
# Bin predictions into 20 buckets
df['bucket'] = pd.cut(df['p_a'], bins=20, labels=False) / 20 + 0.025

cal = df.groupby('bucket').agg(
    mean_pred=('p_a', 'mean'),
    mean_actual=('a_is_winner', 'mean'),
    n=('a_is_winner', 'count'),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.scatter(cal['mean_pred'], cal['mean_actual'], s=cal['n']/10, alpha=0.6, c='C0', edgecolors='black')
ax.set_xlabel('Predicted probability')
ax.set_ylabel('Actual win rate')
ax.set_title('Calibration Curve (point size = number of predictions)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Accuracy over time

How does the model perform year by year? You should see ~67% in Big Three era, declining ~63% post-2018.

In [ ]:
yearly = df.groupby('year').agg(
    accuracy=('correct', 'mean'),
    n=('correct', 'count'),
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly['year'], yearly['accuracy'], color='steelblue', edgecolor='black')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random (50%)')
ax.axhline(y=0.68, color='green', linestyle='--', alpha=0.5, label='Literature Elo (~68%)')
ax.set_xlabel('Year')
ax.set_ylabel('Accuracy')
ax.set_title('Backtest Accuracy by Year')
ax.set_ylim(0.5, 0.75)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Accuracy by surface

Hard usually best (most matches), Grass usually hardest (fewest matches per year).

In [ ]:
surface_perf = df.groupby('surface').agg(
    accuracy=('correct', 'mean'),
    n=('correct', 'count'),
).reset_index().sort_values('n', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(surface_perf['surface'], surface_perf['accuracy'],
              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(surface_perf)],
              edgecolor='black')
for bar, n in zip(bars, surface_perf['n']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'n={n:,}', ha='center', fontsize=10)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('Accuracy')
ax.set_title('Backtest Accuracy by Surface')
ax.set_ylim(0.5, 0.75)
plt.tight_layout()
plt.show()

## 6. Distribution of model confidence

How often is the model 'sure' vs 'unsure'? A healthy distribution has weight across all buckets.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['p_winner'], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('P(actual winner wins) - model probability')
ax.set_ylabel('Count')
ax.set_title('Distribution of model confidence on the eventual winner\n'
             '(Higher = model thought winner was more likely to win)')
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50/50 toss-up')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\nMatches where model thought winner had <25% chance: {(df['p_winner'] < 0.25).sum():,}")
print(f"Matches where model thought winner had >75% chance: {(df['p_winner'] > 0.75).sum():,}")

## 7. Brier score over time (smaller is better)

In [ ]:
df['brier'] = (df['p_a'] - df['a_is_winner']) ** 2
monthly = df.copy()
monthly['month'] = monthly['date'].astype(str).str[:7]
monthly_brier = monthly.groupby('month').agg(brier=('brier', 'mean'),
                                              n=('brier', 'count')).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_brier['month'], monthly_brier['brier'], marker='.', linewidth=1, alpha=0.7)
ax.axhline(y=0.25, color='red', linestyle='--', alpha=0.5, label='Random (0.25)')
ax.set_xlabel('Month')
ax.set_ylabel('Brier score')
ax.set_title('Brier Score Over Time (lower is better)')
ax.set_xticks(monthly_brier['month'][::12])
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()